In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import pickle
import os


def load_data(path='data.parquet'):
    df = pd.read_parquet(path)
    df.index = pd.to_datetime(df.index)
    df = df.ffill()
    return df

# load the data
df = load_data()

In [ ]:
# calculate spread and pnl
df['spread'] = df['banknifty'] - df['nifty']
df['pnl'] = df['spread'] * (df['tte'] ** 0.7)


Grid search for Finding Optimal parameter


In [ ]:
def grid_search(df):
    entry_z_list = np.arange(1.5, 3.1, 0.2)
    exit_z_list = np.arange(0, 1.1, 0.2)
    window_list = [30, 60, 90, 120, 150]
    results = []

    for window in tqdm(window_list, desc="Window"):
        spread_mean = df['spread'].rolling(window).mean()
        spread_std = df['spread'].rolling(window).std()
        zscore = (df['spread'] - spread_mean) / spread_std

        for entry_z in entry_z_list:
            for exit_z in exit_z_list:
                position = 0
                positions = np.zeros(len(df))
                for i in range(window, len(df)):
                    if position == 0:
                        if zscore.iloc[i] > entry_z:
                            position = -1
                        elif zscore.iloc[i] < -entry_z:
                            position = 1
                    elif position == 1 and zscore.iloc[i] >= exit_z:
                        position = 0
                    elif position == -1 and zscore.iloc[i] <= -exit_z:
                        position = 0
                    positions[i] = position

                strategy_pnl = np.roll(positions, 1) * df['pnl'].values
                strategy_pnl[0] = 0
                cum_strategy_pnl = np.cumsum(strategy_pnl)
                temp_df = df.copy()
                temp_df['strategy_pnl'] = strategy_pnl
                daily_pnl = temp_df['strategy_pnl'].resample('D').sum()
                daily_pnl = daily_pnl[daily_pnl != 0]
                sharpe = daily_pnl.mean() / daily_pnl.std() * np.sqrt(252) if daily_pnl.std() > 0 else 0
                abs_return = cum_strategy_pnl[-1]
                rolling_max = np.maximum.accumulate(cum_strategy_pnl)
                drawdown = cum_strategy_pnl - rolling_max
                max_drawdown = drawdown.min()
                results.append({
                    'window': window,
                    'entry_z': entry_z,
                    'exit_z': exit_z,
                    'abs_return': abs_return,
                    'sharpe': sharpe,
                    'max_drawdown': max_drawdown
                })

    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values(by='sharpe', ascending=False)
    return results_df
# Cache results to avoid recomputation
cache_path = 'grid_search_results.pkl'
if os.path.exists(cache_path) and os.path.getsize(cache_path) > 0:
    with open(cache_path, 'rb') as f:
        results_df = pickle.load(f)
    print("Loaded grid search results from cache.")
else:
    results_df = grid_search(df)
    with open(cache_path, 'wb') as f:
        pickle.dump(results_df, f)
    print("Computed and cached grid search results.")

results_df.head(10)

In [ ]:
optimal = results_df.iloc[0]
window = int(optimal['window']) # 90
entry_z = optimal['entry_z'] # 1.5
exit_z = optimal['exit_z'] # 0.8

# Calculate rolling mean and std for spread
df['spread_mean'] = df['spread'].rolling(window).mean()
df['spread_std'] = df['spread'].rolling(window).std()
df['zscore'] = (df['spread'] - df['spread_mean']) / df['spread_std']

# Initialize position and PnL columns
df['position'] = 0  # 1 for long, -1 for short, 0 for flat

# Generate signals
position = 0
for i in range(window, len(df)):
    if position == 0:
        if df['zscore'].iloc[i] > entry_z:
            position = -1  # Short spread
        elif df['zscore'].iloc[i] < -entry_z:
            position = 1   # Long spread
    elif position == 1 and df['zscore'].iloc[i] >= exit_z:
        position = 0
    elif position == -1 and df['zscore'].iloc[i] <= -exit_z:
        position = 0
    df.at[df.index[i], 'position'] = position

# Calculate strategy PnL 
df['strategy_pnl'] = df['position'].shift().fillna(0) * df['pnl']
df['cum_pnl'] = df['strategy_pnl'].cumsum()
df[['spread', 'zscore', 'position', 'strategy_pnl', 'cum_pnl']].tail()

In [ ]:
# Calculate performance metrics
absolute_return = df['cum_pnl'].iloc[-1]

# Sharpe Ratio
daily_pnl = df['strategy_pnl'].resample('D').sum()
daily_pnl = daily_pnl[daily_pnl != 0]
sharpe_ratio = daily_pnl.mean() / daily_pnl.std() * np.sqrt(252) if daily_pnl.std() > 0 else 0

# Max Drawdown and Max Drawdown %
cum_pnl = df['cum_pnl'].fillna(0)
rolling_max = cum_pnl.cummax()
drawdown = cum_pnl - rolling_max
max_drawdown = drawdown.min()
max_drawdown_pct = (max_drawdown / rolling_max.max()) * 100 if rolling_max.max() != 0 else np.nan

# Win Rate
trade_pnls = df['strategy_pnl'][df['strategy_pnl'] != 0]
wins = (trade_pnls > 0).sum()
losses = (trade_pnls < 0).sum()
win_rate = wins / (wins + losses) if (wins + losses) > 0 else np.nan

# Trade Count
trade_count = (df['position'].diff().abs() > 0).sum()

print(f"Absolute Return: {absolute_return:.2f}")
print(f"Annualized Sharpe Ratio: {sharpe_ratio:.2f}")
print(f"Max Drawdown: {max_drawdown:.2f}")
print(f"Max Drawdown %: {max_drawdown_pct:.2f}%")
print(f"Win Rate: {win_rate:.2%}")
print(f"Trade Count: {trade_count}")

In [ ]:
import matplotlib.pyplot as plt

plt.style.use('dark_background')
fig, ax = plt.subplots(figsize=(16, 7))
ax.plot(df.index, df['cum_pnl'], label='Cumulative Strategy PnL', color='green', linewidth=2)
ax.set_title('Cumulative (Absolute) Strategy PnL', fontsize=18, fontweight='bold')
ax.set_xlabel('Date', fontsize=14)
ax.set_ylabel('Cumulative PnL', fontsize=14)
ax.legend(fontsize=12)
ax.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()